# Setup From GitHub

Set `REPO_URL` to your GitHub repo. These notebooks avoid reinstalling PyTorch because Kaggle already ships a CUDA-compatible build.

In [ ]:
REPO_URL = "https://github.com/ahmad903501/qwen-dpo-alignment.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/qwen-dpo-alignment"


In [ ]:
from pathlib import Path

repo_path = Path(REPO_DIR)
if not repo_path.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only origin {BRANCH}

%cd {REPO_DIR}
!pwd


In [ ]:
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps

import torch, transformers, datasets
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())
!nvidia-smi


# Generation Only

Use this notebook to generate AlpacaEval-subset outputs from base, SFT, and DPO models. If the checkpoints were trained in another Kaggle notebook, attach that notebook's outputs as an input dataset and set `SFT_MODEL_PATH` / `DPO_MODEL_PATH` accordingly.

In [ ]:
SUBSET_SIZE = 300
SEED = 42
BASE_MODEL = "Qwen/Qwen2.5-0.5B"

# If running generation in the same notebook/session as training, these paths work:
SFT_MODEL_PATH = "outputs/sft-qwen-0.5b/best"
DPO_MODEL_PATH = "outputs/dpo-qwen-0.5b/best"

# If using a separate Kaggle notebook with attached outputs, use paths like:
# SFT_MODEL_PATH = "/kaggle/input/YOUR_TRAINING_OUTPUT_DATASET/outputs/sft-qwen-0.5b/best"
# DPO_MODEL_PATH = "/kaggle/input/YOUR_TRAINING_OUTPUT_DATASET/outputs/dpo-qwen-0.5b/best"

SUBSET_PATH = f"data/alpaca_eval/alpacaeval_{SUBSET_SIZE}_seed{SEED}.json"


## Create AlpacaEval Subset

In [ ]:
!python scripts/make_alpaca_subset.py \
  --size {SUBSET_SIZE} \
  --seed {SEED} \
  --output_dir data/alpaca_eval


## Generate Base Outputs

In [ ]:
!python scripts/generate_alpaca.py \
  --model_name_or_path {BASE_MODEL} \
  --subset_path {SUBSET_PATH} \
  --output_path outputs/eval/base_alpacaeval_{SUBSET_SIZE}.json \
  --generator_name qwen2.5-0.5b-base \
  --batch_size 8 \
  --max_new_tokens 512


## Generate SFT Outputs

In [ ]:
!python scripts/generate_alpaca.py \
  --model_name_or_path {SFT_MODEL_PATH} \
  --subset_path {SUBSET_PATH} \
  --output_path outputs/eval/sft_alpacaeval_{SUBSET_SIZE}.json \
  --generator_name qwen2.5-0.5b-sft \
  --batch_size 8 \
  --max_new_tokens 512


## Generate DPO Outputs

In [ ]:
!python scripts/generate_alpaca.py \
  --model_name_or_path {DPO_MODEL_PATH} \
  --subset_path {SUBSET_PATH} \
  --output_path outputs/eval/dpo_alpacaeval_{SUBSET_SIZE}.json \
  --generator_name qwen2.5-0.5b-dpo \
  --batch_size 8 \
  --max_new_tokens 512


## Build Pairwise Files For Evaluation

In [ ]:
!python scripts/build_pairwise_eval.py \
  --a_outputs outputs/eval/dpo_alpacaeval_{SUBSET_SIZE}.json \
  --b_outputs outputs/eval/sft_alpacaeval_{SUBSET_SIZE}.json \
  --a_name dpo \
  --b_name sft \
  --output_path outputs/eval/dpo_vs_sft_pairs.jsonl

!python scripts/build_pairwise_eval.py \
  --a_outputs outputs/eval/dpo_alpacaeval_{SUBSET_SIZE}.json \
  --b_outputs outputs/eval/base_alpacaeval_{SUBSET_SIZE}.json \
  --a_name dpo \
  --b_name base \
  --output_path outputs/eval/dpo_vs_base_pairs.jsonl

!python scripts/build_pairwise_eval.py \
  --a_outputs outputs/eval/sft_alpacaeval_{SUBSET_SIZE}.json \
  --b_outputs outputs/eval/base_alpacaeval_{SUBSET_SIZE}.json \
  --a_name sft \
  --b_name base \
  --output_path outputs/eval/sft_vs_base_pairs.jsonl

!python scripts/build_pairwise_eval.py \
  --a_outputs outputs/eval/dpo_alpacaeval_{SUBSET_SIZE}.json \
  --b_from_reference \
  --a_name dpo \
  --b_name gpt_reference \
  --output_path outputs/eval/dpo_vs_reference_pairs.jsonl


## Check Generation Artifacts

In [ ]:
!find outputs/eval -maxdepth 1 -type f | sort
